In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('data/sales_2023_2025.csv')

In [3]:
df.head()

In [4]:
df.shape

In [5]:
df.dtypes

In [6]:
df.isna().sum()

In [7]:
df['date'] = pd.to_datetime(df['date'])

In [8]:
df.describe()

In [9]:
stores = pd.read_csv('data/store_dim.csv')
stores.head()

In [10]:
stores['region'].value_counts()

In [11]:
df = df.merge(stores, on='store_id', how='left')

In [12]:
df['region'].isna().sum()

In [13]:
df = df[df['region'].notna()]

In [14]:
df['revenue'] = df['units'] * df['unit_price']

In [15]:
df[df['revenue'] < 0].head(20)

those negatives are returns, dropping them for now

In [16]:
df = df[df['revenue'] >= 0]

In [17]:
df.groupby('region')['revenue'].sum().sort_values(ascending=False)

In [18]:
df.groupby('region')['revenue'].sum().plot(kind='bar')
plt.show()

In [19]:
weekly = df.set_index('date').groupby(pd.Grouper(freq='W'))['revenue'].sum()

In [20]:
weekly.plot(figsize=(12,4))
plt.show()

In [21]:
weekly.head(20)

In [22]:
weekly.describe()

In [23]:
by_region = df.set_index('date').groupby([pd.Grouper(freq='W'), 'region'])['revenue'].sum().unstack()

In [24]:
by_region.plot(figsize=(12,5))
plt.show()

In [25]:
by_region.corr()

In [26]:
df['month'] = df['date'].dt.month

In [27]:
pivot = df.pivot_table(index='month', columns='region', values='revenue', aggfunc='sum')

In [28]:
sns.heatmap(pivot, annot=False, cmap='viridis')
plt.show()

In [29]:
df['units'].quantile([.5,.9,.99,.999])

In [30]:
df.loc[df['units'] > 1000].head()

In [34]:
df = df[df['units'] <= 1000]

In [35]:
weekly = df.set_index('date').groupby(pd.Grouper(freq='W'))['revenue'].sum()
weekly.plot(figsize=(12,4))
plt.show()

In [36]:
df.groupby('region')['revenue'].sum().sort_values(ascending=False)

In [31]:
tmp = df[df['region'] == 'NORTH']
tmp['revenue'].hist(bins=50)
plt.show()

In [32]:
tmp2 = df[df['region'] == 'SOUTH']
tmp2['revenue'].hist(bins=50)
plt.show()

In [33]:
del tmp, tmp2

In [37]:
agg = df.groupby(['store_id', pd.Grouper(key='date', freq='W')])['revenue'].sum().reset_index()

In [38]:
agg['lag_1'] = agg.groupby('store_id')['revenue'].shift(1)
agg['lag_4'] = agg.groupby('store_id')['revenue'].shift(4)

In [39]:
agg['roll_4'] = agg.groupby('store_id')['revenue'].transform(lambda s: s.rolling(4).mean())

In [40]:
agg = agg.dropna()

In [41]:
agg.shape

In [42]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error

In [43]:
features = ['lag_1','lag_4','roll_4']
X = agg[features]
y = agg['revenue']

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [45]:
model = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05)
model.fit(X_train, y_train)

In [46]:
preds = model.predict(X_test)

In [47]:
mean_absolute_percentage_error(y_test, preds)

In [48]:
plt.figure(figsize=(12,4))
plt.plot(y_test.values[:200], label='actual')
plt.plot(preds[:200], label='pred')
plt.legend()
plt.show()

In [49]:
pd.Series(model.feature_importances_, index=features).sort_values()

In [50]:
model2 = GradientBoostingRegressor(n_estimators=800, max_depth=5, learning_rate=0.02)
model2.fit(X_train, y_train)
preds2 = model2.predict(X_test)
mean_absolute_percentage_error(y_test, preds2)

In [51]:
residuals = y_test.values - preds
plt.hist(residuals, bins=60)
plt.show()

In [52]:
agg['pred'] = np.nan
agg.loc[X_test.index, 'pred'] = preds

In [53]:
out = agg[['store_id','date','revenue','pred']]
out.to_csv('forecast_v4.csv', index=False)

In [54]:
out.head()

In [ ]:
# TODO try lightgbm
# import lightgbm as lgb

In [ ]:
import joblib
joblib.dump(model, 'model_v4.pkl')